In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# 1. Retrieve Atlas Connection Secrets from Azure Key Vault
SECRET_SCOPE = "maven-secrets"
MONGO_URI = dbutils.secrets.get(scope=SECRET_SCOPE, key="mongodb-uri")
DATABASE_NAME = "maven_market_db"
COLLECTION_NAME = "customers"

In [0]:
#2. Ingest Customer Data from MongoDB Atlas
df_customers_raw = (
    spark.read.format("mongodb")
    .option("spark.mongodb.read.connection.uri", MONGO_URI)
    .option("spark.mongodb.read.database", "maven_market")
    .option("spark.mongodb.read.collection", "customers")
    .load()
)

In [0]:
# 3. Add Lineage & Ingestion Metadata
df_customers_bronze = (
    df_customers_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_system", lit("MongoDB_Atlas"))
    .withColumn("_source_collection", lit(COLLECTION_NAME))
)

In [0]:
display(df_customers_bronze)

In [0]:
catalog_name = 'mavenmarket'
schema_name = 'bronze'
table_name = 'customers'

In [0]:
# 4. Save to Bronze Delta Table (Full-load overwrite for master snapshot)
full_table_path = f"{catalog_name}.{schema_name}.{table_name}"

(
    df_customers_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_path)
)
